# T04 CNN MNIST và Gender

MNIST dùng ba khối tích chập và hai lớp Dense 512. Gender dùng năm khối tích chập, hai lớp Dense 1024 và xoay ảnh NPY 90° theo chiều kim đồng hồ như notebook gốc. Loss validation điều khiển EarlyStopping và giảm learning rate.

Chạy từ môi trường cài requirements. Đổi cấu hình và tên lượt chạy nếu muốn thí nghiệm mới. Notebook dùng kết quả đã lưu nếu lượt chạy đã hoàn tất.

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "src" / "train_cnn.py").is_file():
    raise RuntimeError("Mở notebook từ thư mục BTL03 hoặc notebooks")
sys.path.insert(0, str(root))
from src.train_cnn import train_cnn
import os
os.chdir(root)

configs = {
    "mnist": dict(data_dir=Path("data/mnist"), epochs=4, optimizer_name="adam", lr=0.001),
    "gender": dict(data_dir=Path("data/gender"), epochs=6, threads=3),
}
run_suffix = "cpu_seed42"

In [ ]:
runs = {}
for dataset, config in configs.items():
    run = Path("results") / f"{dataset}_{run_suffix}"
    if not (run / "metrics.json").is_file():
        if dataset == "gender" and len(list(config["data_dir"].glob("*.npy"))) < 4:
            print("Gender: cần bốn file NPY của giảng viên trong data/gender.")
            continue
        train_cnn(dataset, output_dir=run, device="cpu", **config)
    runs[dataset] = run
    print(json.loads((run / "metrics.json").read_text(encoding="utf-8")))

In [ ]:
import csv
import matplotlib.pyplot as plt

for dataset, run in runs.items():
    with (run / "history.csv").open() as file:
        history = list(csv.DictReader(file))
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    for metric, ax in zip(("loss", "accuracy"), axes):
        for split in ("train", "val"):
            ax.plot([int(row["epoch"]) for row in history],
                    [float(row[f"{split}_{metric}"]) for row in history],
                    marker="o", label=split)
        ax.set(xlabel="Epoch", ylabel=metric, title=dataset)
        ax.legend()
    fig.tight_layout()
    fig.savefig(run / "learning_curve.png", dpi=160)
    plt.show()